# 08E — Reviewer diagnostics: selection, fire concentration, and simple prediction models

This notebook examines three questions raised during manuscript review:
1. How do NIROPS observation coverage and the existing inclusion rules change the historical population?
2. How much do fires with many mapped fireline units contribute to the reported results?
3. Do local predictor values improve predictions beyond simple local relationships and conditions shared within a fire?

**Run all cells from the top using the project's `.venv` Python kernel in Antigravity.** The notebook is self-contained and reads the pinned completed runs below. It writes a new `08E_...` output directory, saves each model/split separately, and can resume verified checkpoints. It does not rerun LightGBM's original 803-predictor training or change earlier results.

All analyses concern the **retrospectively selected historical population**. The diagnostics were designed after earlier results were examined and are exploratory. They do not independently validate outcome labels, establish construction times, or demonstrate performance during ongoing fires.

The principal errors give every fireline unit equal weight. Matched equal-fire results are also reported. Neither weighting is selected according to which result favors a model.

## 0. Configuration and prerequisites

The default is the full analysis: three new model procedures, each fitted on five saved fire-held-out splits and four chronological splits (**27 fits**). CPU threads are bounded. No tuning, calibration, or threshold search is performed.

After an interrupted run, set `RESUME_RUN_DIR` to the printed output directory and run from the top. The notebook rejects checkpoints if code, inputs, settings, or splits differ. Do not change `IS_SMOKE_TEST` for the scientific run; the small verification mode produces explicitly marked diagnostic outputs.

In [ ]:
import os  # Configurable project root for the public release.
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version as package_version
import os, sys, json, hashlib, uuid, time, platform, warnings, ast, copy
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display, Markdown
import joblib

PROJECT_ROOT_OVERRIDE = None  # e.g. Path(r"PROJECT_ROOT")
RESUME_RUN_DIR = None  # paste an existing 08E output-directory path to resume
IS_SMOKE_TEST = False
BOOTSTRAP_REPLICATES = 2000
RANDOM_SEED = 20261105
N_JOBS = min(8, max(1, (os.cpu_count() or 2) // 2))
SMOKE_UNITS_PER_FIRE = 20
BUILD_ID = "08E_V1_REVIEWER_DIAGNOSTICS"

def find_root():
    candidates = ([Path(PROJECT_ROOT_OVERRIDE)] if PROJECT_ROOT_OVERRIDE else []) + [Path.cwd(), *Path.cwd().parents]
    candidates += [Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))]
    for candidate in candidates:
        if (candidate / "data/processed/modeling_table_assembly_v1/fireline_hold_modeling_master_v1.parquet").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Set PROJECT_ROOT_OVERRIDE to the project folder, then Run All.")

PROJECT_ROOT = find_root()
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/08E_reviewer_diagnostics.ipynb"
B_DIR = PROJECT_ROOT / "outputs/08_condition_outcomes/08B_20260930T224636_898345Z_38d03c"
D_DIR = PROJECT_ROOT / "outputs/08_condition_outcomes/08D_20261001T002458_892164Z_f3e697"
UNIT_PATH = B_DIR / "08B_existing_units_with_diagnostic_weights.parquet"
MASTER_PATH = PROJECT_ROOT / "data/processed/modeling_table_assembly_v1/fireline_hold_modeling_master_v1.parquet"
OOF_PATH = PROJECT_ROOT / "data/processed/baseline_modeling_v1/oof_predictions/E_FULL_EXTENDED__row.parquet"
FORWARD_PATH = PROJECT_ROOT / "data/processed/forward_temporal_generalization_v1/FORWARD_TEMPORAL_PREDICTIONS_V1.parquet"
if IS_SMOKE_TEST:
    BOOTSTRAP_REPLICATES = 40
    N_JOBS = min(N_JOBS, 2)
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 25)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 300})
print("Project:", PROJECT_ROOT)
print("Mode:", "SOFTWARE VERIFICATION ONLY" if IS_SMOKE_TEST else "FULL ANALYSIS", "| CPU threads:", N_JOBS)

## 1. Utilities, provenance, and checks

Input fingerprints include file contents, notebook source, package versions, and scientific settings. A completed marker is written only after every phase passes. Checkpoints are reusable only within an identical run specification.

In [ ]:
def require(ok, message):
    if not bool(ok): raise AssertionError(message)

def need(*names):
    missing = [name for name in names if name not in phase_done]
    require(not missing, "Run preceding sections first: " + ", ".join(missing))

def safe_json(x):
    if isinstance(x, dict): return {str(k): safe_json(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [safe_json(v) for v in x]
    if isinstance(x, np.ndarray): return safe_json(x.tolist())
    if isinstance(x, (Path, pd.Timestamp)): return str(x)
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, (float, np.floating)): return float(x) if np.isfinite(x) else None
    return x

def write_json(path, value):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(safe_json(value), indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temp, path)

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""): h.update(chunk)
    return h.hexdigest()

def notebook_source_fingerprint(doc):
    normalized=[]
    for cell in doc["cells"]:
        text="".join(cell["source"]) if isinstance(cell["source"],list) else cell["source"]
        if cell["cell_type"]=="code":
            tree=ast.parse(text)
            # Resume location is operational, not a scientific setting. Keep all
            # other assignments and behavior in the source fingerprint.
            for node in tree.body:
                if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and node.targets[0].id=="RESUME_RUN_DIR":
                    node.value=ast.Constant(value="__OPERATIONAL_RESUME_PATH__")
            normalized.append(["code",ast.dump(tree,include_attributes=False)])
        else:
            normalized.append([cell["cell_type"],text])
    return hashlib.sha256(json.dumps(normalized,ensure_ascii=False).encode()).hexdigest()

def source_hash():
    return notebook_source_fingerprint(json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8")))

def frame_hash(frame, columns):
    f = frame.loc[:, columns].sort_values("unit_uid").reset_index(drop=True)
    h = hashlib.sha256(json.dumps([(c, str(f[c].dtype)) for c in f], sort_keys=True).encode())
    h.update(pd.util.hash_pandas_object(f, index=False).to_numpy().tobytes())
    return h.hexdigest()

tables = {}
phase_done = set()
figure_files = []

## 2. Selection-diagnostic definitions

The function below reads saved candidate fireline units and saved selection memberships. Full-record observation counts describe the selection process; they are **never model predictors**. The preselection denominator is the corrected Held/Burned Over candidate population already processed through upstream NIFC–NIROPS matching. It does not represent every NIFC fireline or every wildfire.

Counts of source records and counts of fireline units answer different questions and are kept separate. Saved mapping dates remain mapping metadata, not verified construction dates.

In [ ]:
"""Read-only selection/observation diagnostics for the frozen fireline study.

The denominator is the saved 03M corrected-unit universe, not all NIFC records
and not the final model cohort. No labels, evidence rules, or membership change.
The notebook builder embeds this module; this module itself writes no outputs.
"""

from pathlib import Path
import numpy as np
import pandas as pd


_SELECTION_BASE = "data/processed/fireline_corrected_local_250m_10m"
SELECTION_RELATIVE_PARQUETS = {
    "candidates": _SELECTION_BASE + "/unit_evidence_rule_audit/weighted_unit_evidence_metrics.parquet",
    "supported": _SELECTION_BASE + "/frozen_cohort/fireline_ml_cohort_frozen_v1.parquet",
    "timing": _SELECTION_BASE + "/frozen_cohort/fireline_ml_dynamic_timing_cohort_frozen_v1.parquet",
    "excluded": _SELECTION_BASE + "/frozen_cohort/fireline_ml_excluded_units_v1.parquet",
}


def selection_input_paths(project_root):
    """Existing files the diagnostic reads, for the caller's input fingerprint.

    If the upstream universe is unavailable, final-fire timestamps can still be
    described. The final master then supplies only the fire IDs for this list;
    it is not substituted as an inclusion-rate denominator.
    """
    root = Path(project_root)
    paths = [root / rel for rel in SELECTION_RELATIVE_PARQUETS.values()
             if (root / rel).is_file()]
    candidate = root / SELECTION_RELATIVE_PARQUETS["candidates"]
    if candidate.is_file():
        ids = pd.read_parquet(candidate, columns=["_irwin_norm"])["_irwin_norm"]
    else:
        master = root / "data/processed/modeling_table_assembly_v1/fireline_hold_modeling_master_v1.parquet"
        if not master.is_file():
            return sorted(paths, key=str)
        paths.append(master)
        ids = pd.read_parquet(master, columns=["_irwin_norm"])["_irwin_norm"]
    trajectory_root = root / _SELECTION_BASE / "trajectories"
    paths.extend(trajectory_root / (str(fire) + ".npz")
                 for fire in sorted(ids.dropna().astype(str).unique())
                 if (trajectory_root / (str(fire) + ".npz")).is_file())
    return sorted(set(paths), key=str)


def _selection_ns(values):
    idx = pd.DatetimeIndex(pd.to_datetime(values, utc=True))
    return np.asarray(idx.tz_localize(None), dtype="datetime64[ns]").view(np.int64)


def _selection_decode_times(raw):
    """Use the 03N magnitude decoder, not the misleading cache-key suffix."""
    raw = np.asarray(raw)
    if raw.ndim != 1 or len(raw) < 2 or not np.issubdtype(raw.dtype, np.integer):
        raise AssertionError("Expected a one-dimensional integer NIROPS timestamp array.")
    raw = raw.astype(np.int64, copy=False)
    if (raw == np.iinfo(np.int64).min).any():
        raise AssertionError("Missing NIROPS timestamp in the trajectory cache.")
    magnitude = float(np.median(np.abs(raw.astype(np.float64))))
    scale = "ns" if magnitude >= 1e17 else "us" if magnitude >= 1e14 else "ms" if magnitude >= 1e11 else "s"
    times = pd.DatetimeIndex(pd.to_datetime(raw, unit=scale, utc=True))
    ns = _selection_ns(times)
    if not np.all(np.diff(ns) > 0):
        raise AssertionError("Cached NIROPS observation times must be strictly increasing.")
    if not ((times.year >= 2019) & (times.year <= 2025)).all():
        raise AssertionError("Unexpected cached NIROPS year; verify the timestamp scale.")
    gap_h = np.diff(ns) / 3.6e12
    if not (0.25 <= float(np.median(gap_h)) <= 240):
        raise AssertionError("Implausible median NIROPS interval after timestamp decoding.")
    return times, ns, scale


def _selection_count_bin(values):
    return (pd.cut(pd.to_numeric(values, errors="coerce"),
                   [-np.inf, 0, 2, 5, 10, 20, 40, np.inf],
                   labels=["0", "1-2", "3-5", "6-10", "11-20", "21-40", "41+"])
            .astype("string").fillna("Unavailable"))


def _selection_stage_table(frame, group_columns, scope, stages):
    """Cumulative and conditional retention, preserving explicit denominators."""
    keys = list(group_columns)
    gb = frame.groupby(keys, dropna=False, observed=True, sort=True) if keys else [((), frame)]
    rows = []
    for key, group in gb:
        key = key if isinstance(key, tuple) else (key,)
        identity = dict(zip(keys, key))
        candidate_n = len(group)
        previous_n = candidate_n
        for order, (stage, flag) in enumerate(stages):
            kept = group if flag is None else group.loc[group[flag]]
            n_held = int(kept["recorded_outcome"].eq("Held").sum())
            rows.append({
                "scope": scope, **identity, "stage_order": order, "stage": stage,
                "candidate_fireline_units": candidate_n,
                "candidate_fires": group["_irwin_norm"].nunique(),
                "previous_stage_fireline_units": previous_n,
                "retained_fireline_units": len(kept),
                "retained_fires": kept["_irwin_norm"].nunique(),
                "excluded_since_previous_stage": previous_n - len(kept),
                "retained_fraction_of_candidates": len(kept) / candidate_n if candidate_n else np.nan,
                "retained_fraction_of_previous_stage": len(kept) / previous_n if previous_n else np.nan,
                "retained_held": n_held,
                "retained_burned_over": len(kept) - n_held,
                "retained_unit_held_fraction": n_held / len(kept) if len(kept) else np.nan,
                "candidate_unit_held_fraction": group["recorded_outcome"].eq("Held").mean(),
                "denominator_scope": "572200 corrected candidate fireline units in 415 production fires; not the full NIFC archive",
            })
            previous_n = len(kept)
    return pd.DataFrame(rows)


def run_selection_diagnostics(project_root, units):
    """Return diagnostic DataFrames; never write, refit, or change selection.

    ``units`` is the complete frozen 304143-row model population, even in a
    notebook software check that samples observations for its model fits.
    Observation counts use all cached NIROPS maps for the respective fire.
    Pre/post-cutoff summaries are conditional on final inclusion, because an
    assigned prediction time is not defined for every excluded candidate.
    """
    root = Path(project_root)
    required = ["unit_uid", "_irwin_norm", "source_record_key", "Year", "outer_fold", "y", "prediction_anchor_utc"]
    missing = sorted(set(required) - set(units.columns))
    assert not missing, "Selection diagnostics require columns: " + str(missing)
    assert len(units) == 304143 and units["_irwin_norm"].nunique() == 381, "Pass the FULL frozen population, not a software-check model sample."
    assert units["unit_uid"].notna().all() and units["unit_uid"].is_unique
    assert units["source_record_key"].notna().all() and units["y"].isin([0, 1]).all()
    final = units[required].copy().reset_index(drop=True)
    final["Year"] = pd.to_numeric(final["Year"], errors="raise").astype(int)
    final["prediction_anchor_utc"] = pd.to_datetime(final["prediction_anchor_utc"], utc=True, errors="raise")
    assert final["prediction_anchor_utc"].notna().all()
    assert final.groupby("_irwin_norm")["Year"].nunique().eq(1).all()
    final["recorded_outcome"] = np.where(final["y"].eq(1), "Held", "Burned Over")
    paths = {name: root / rel for name, rel in SELECTION_RELATIVE_PARQUETS.items()}
    results = {}
    status = []

    def note(diagnostic, state, detail):
        status.append({"diagnostic": diagnostic, "status": state, "detail": detail})

    candidate = None
    supported = None
    if paths["supported"].is_file():
        supported = pd.read_parquet(paths["supported"], columns=[
            "unit_uid", "_irwin_norm", "Year", "hold_label", "earliest_supported_start_idx",
            "earliest_supported_interval_start_utc_timing", "dynamic_timing_eligible"])
        assert supported["unit_uid"].is_unique and len(supported) == 317495
    if paths["candidates"].is_file():
        candidate = pd.read_parquet(paths["candidates"], columns=[
            "unit_uid", "_irwin_norm", "Year", "IncidentName", "_engagement_norm",
            "weighted_support_fraction", "weighted_conflict_fraction", "actual_line_length_m"])
        assert len(candidate) == 572200 and candidate["_irwin_norm"].nunique() == 415
        assert candidate["unit_uid"].notna().all() and candidate["unit_uid"].is_unique
        assert candidate["_engagement_norm"].isin(["Held", "Burned Over"]).all()
        candidate["Year"] = pd.to_numeric(candidate["Year"], errors="raise").astype(int)
        assert candidate["Year"].between(2020, 2024).all()
        assert candidate.groupby("_irwin_norm")["Year"].nunique().eq(1).all()
        candidate = candidate.rename(columns={"_engagement_norm": "recorded_outcome"})
        assert final["unit_uid"].isin(candidate["unit_uid"]).all()
        check = final[["unit_uid", "_irwin_norm", "Year", "recorded_outcome"]].merge(
            candidate[["unit_uid", "_irwin_norm", "Year", "recorded_outcome"]],
            on="unit_uid", validate="one_to_one", suffixes=("_final", "_candidate"))
        for name in ["_irwin_norm", "Year", "recorded_outcome"]:
            assert check[name + "_final"].eq(check[name + "_candidate"]).all(), name
        note("candidate_denominator", "available", "572200 corrected fireline units / 415 fires before the 03N support filter; already subject to upstream matching/production eligibility.")
    else:
        note("candidate_denominator", "unavailable", "Missing saved 03M corrected-unit universe. No inclusion rates will use the final cohort as a substitute denominator.")

    stages = [("corrected_candidates", None)]
    stage_files_ok = candidate is not None and supported is not None and paths["timing"].is_file() and paths["excluded"].is_file()
    if stage_files_ok:
        timing = pd.read_parquet(paths["timing"], columns=["unit_uid"])
        excluded = pd.read_parquet(paths["excluded"], columns=["unit_uid", "final_exclusion_reason"])
        assert timing["unit_uid"].is_unique and len(timing) == 315035
        assert excluded["unit_uid"].is_unique and len(excluded) == 254705
        assert supported["unit_uid"].isin(candidate["unit_uid"]).all()
        assert excluded["unit_uid"].isin(candidate["unit_uid"]).all()
        assert not excluded["unit_uid"].isin(supported["unit_uid"]).any()
        assert len(excluded) + len(supported) == len(candidate)
        assert timing["unit_uid"].isin(supported["unit_uid"]).all()
        assert final["unit_uid"].isin(timing["unit_uid"]).all()
        assert set(supported.loc[supported["dynamic_timing_eligible"], "unit_uid"]) == set(timing["unit_uid"])
        candidate["retained_support"] = candidate["unit_uid"].isin(supported["unit_uid"])
        candidate["retained_timing"] = candidate["unit_uid"].isin(timing["unit_uid"])
        candidate["retained_final"] = candidate["unit_uid"].isin(final["unit_uid"])
        stages += [("support_and_held_conflict_rules", "retained_support"),
                   ("timing_coherence", "retained_timing"), ("final_modeling_cohort", "retained_final")]
        candidate["selection_status"] = "final_modeling_cohort"
        reason = excluded.set_index("unit_uid")["final_exclusion_reason"]
        m = ~candidate["retained_support"]
        candidate.loc[m, "selection_status"] = candidate.loc[m, "unit_uid"].map(reason)
        candidate.loc[candidate["retained_support"] & ~candidate["retained_timing"], "selection_status"] = "support_retained_but_not_timing_eligible"
        candidate.loc[candidate["retained_timing"] & ~candidate["retained_final"], "selection_status"] = "timing_eligible_but_not_in_final_modeling_cohort"
        assert candidate["selection_status"].notna().all()
        results["selection_exclusion_reason_counts"] = candidate.groupby(
            ["Year", "recorded_outcome", "selection_status"], observed=True, dropna=False).agg(
            fireline_units=("unit_uid", "size"), fires=("_irwin_norm", "nunique")).reset_index()
        results["selection_stage_counts"] = pd.concat([
            _selection_stage_table(candidate, [], "overall", stages),
            _selection_stage_table(candidate, ["Year"], "year", stages),
            _selection_stage_table(candidate, ["recorded_outcome"], "outcome", stages),
            _selection_stage_table(candidate, ["Year", "recorded_outcome"], "year_outcome", stages),
        ], ignore_index=True)
        note("selection_stage_membership", "available", "Counts use saved support/timing/final memberships, without rerunning evidence rules or changing labels.")
    else:
        absent = [name for name, path in paths.items() if not path.is_file()]
        note("selection_stage_membership", "unavailable", "Required upstream stage file(s) unavailable: " + ", ".join(absent))
        results["selection_stage_counts"] = pd.DataFrame()
        results["selection_exclusion_reason_counts"] = pd.DataFrame()

    coverage_rows = []
    per_unit_rows = []
    fire_frame = candidate if candidate is not None else final
    final_by_fire = {fire: frame for fire, frame in final.groupby("_irwin_norm", sort=False)}
    supported_lookup = supported.set_index("unit_uid") if supported is not None else None
    fires = sorted(fire_frame["_irwin_norm"].astype(str).unique())
    for position, fire in enumerate(fires, start=1):
        path = root / _SELECTION_BASE / "trajectories" / (fire + ".npz")
        if not path.is_file():
            coverage_rows.append({"_irwin_norm": fire, "observation_coverage_status": "missing_trajectory_cache"})
            if fire in final_by_fire:
                f = final_by_fire[fire].copy()
                f["observation_coverage_status"] = "missing_trajectory_cache"
                per_unit_rows.append(f)
            continue
        with np.load(path, allow_pickle=False) as cache:
            # NPZ member access decompresses only timestamps, never the signed-distance matrix.
            assert "times_ns" in cache.files, "Missing cached NIROPS timestamps: " + str(path)
            times, ns, scale = _selection_decode_times(cache["times_ns"])
        assert len(ns) >= 3, "The production fire inventory required at least three NIROPS observations."
        gaps = np.diff(ns) / 3.6e12
        coverage_rows.append({
            "_irwin_norm": fire, "observation_coverage_status": "available",
            "nirops_observation_count": len(ns), "first_nirops_observation_utc": times[0],
            "last_nirops_observation_utc": times[-1],
            "observation_span_hours": (ns[-1] - ns[0]) / 3.6e12,
            "median_observation_gap_hours": float(np.median(gaps)),
            "max_observation_gap_hours": float(np.max(gaps)),
            "consecutive_intervals_at_most_48h": int(np.count_nonzero(gaps <= 48.0)),
            "consecutive_intervals_over_48h": int(np.count_nonzero(gaps > 48.0)),
            "decoded_timestamp_integer_unit": scale,
        })
        if fire in final_by_fire:
            f = final_by_fire[fire].copy()
            anchors = _selection_ns(f["prediction_anchor_utc"])
            left = np.searchsorted(ns, anchors, side="left")
            right = np.searchsorted(ns, anchors, side="right")
            assert (right - left == 1).all(), "Prediction time not exactly one cached observation: " + fire
            assert (right < len(ns)).all(), "Final prediction time lacks a subsequent NIROPS map."
            f["observation_coverage_status"] = "available"
            f["nirops_observation_count"] = len(ns)
            f["observations_before_prediction_time"] = left
            f["observations_at_prediction_time"] = right - left
            f["observations_through_prediction_time"] = right
            f["observations_after_prediction_time"] = len(ns) - right
            f["hours_from_first_observation_to_prediction_time"] = (anchors - ns[0]) / 3.6e12
            f["hours_from_prediction_time_to_last_observation"] = (ns[-1] - anchors) / 3.6e12
            previous = ns[np.maximum(left - 1, 0)]
            f["hours_since_previous_observation"] = np.where(left > 0, (anchors - previous) / 3.6e12, np.nan)
            f["hours_until_next_observation"] = (ns[right] - anchors) / 3.6e12
            f["observations_in_previous_48h_excluding_prediction_time"] = left - np.searchsorted(ns, anchors - int(48 * 3.6e12), side="left")
            f["observations_in_next_48h_excluding_prediction_time"] = np.searchsorted(ns, anchors + int(48 * 3.6e12), side="right") - right
            if supported_lookup is not None:
                mapped = supported_lookup.loc[f["unit_uid"]]
                assert np.array_equal(mapped["earliest_supported_start_idx"].to_numpy(dtype=int), left), "Earliest qualifying interval does not match saved prediction time."
                assert np.array_equal(_selection_ns(mapped["earliest_supported_interval_start_utc_timing"]), anchors)
                f["qualifying_interval_end_utc"] = pd.to_datetime(ns[right], utc=True)
                f["qualifying_interval_hours"] = (ns[right] - anchors) / 3.6e12
                assert f["qualifying_interval_hours"].le(48.0 + 1e-9).all()
                f["observations_after_qualifying_interval"] = len(ns) - right - 1
                f["hours_from_qualifying_interval_end_to_last_observation"] = (ns[-1] - ns[right]) / 3.6e12
                assert f.loc[f["y"].eq(1), "observations_after_qualifying_interval"].ge(1).all()
            per_unit_rows.append(f)
        if position % 50 == 0 or position == len(fires):
            print("Selection diagnostics: timestamp coverage", position, "/", len(fires), "fires")

    coverage = pd.DataFrame(coverage_rows)
    assert coverage["_irwin_norm"].is_unique
    if "nirops_observation_count" not in coverage:
        coverage["nirops_observation_count"] = np.nan
    coverage["observation_count_bin"] = _selection_count_bin(coverage["nirops_observation_count"])
    unavailable = int(coverage["observation_coverage_status"].ne("available").sum())
    note("cached_observation_coverage", "available" if not unavailable else "partial", f"{len(coverage)-unavailable}/{len(coverage)} fires have usable cached time arrays; missing caches remain explicit and are not counted as zero observations.")

    if candidate is not None:
        candidate = candidate.merge(coverage, on="_irwin_norm", how="left", validate="many_to_one")
        assert candidate["observation_count_bin"].notna().all()
        aggregations = {"candidate_fireline_units": ("unit_uid", "size"),
                        "candidate_held": ("recorded_outcome", lambda x: int(x.eq("Held").sum()))}
        if stage_files_ok:
            aggregations.update({"support_retained_fireline_units": ("retained_support", "sum"),
                                 "timing_retained_fireline_units": ("retained_timing", "sum"),
                                 "final_retained_fireline_units": ("retained_final", "sum")})
        fire_stats = candidate.groupby(["_irwin_norm", "Year"], observed=True).agg(**aggregations).reset_index()
        fire_stats["candidate_burned_over"] = fire_stats["candidate_fireline_units"] - fire_stats["candidate_held"]
        fire_stats["candidate_unit_held_fraction"] = fire_stats["candidate_held"] / fire_stats["candidate_fireline_units"]
        if stage_files_ok:
            fire_stats["final_retained_fraction_of_candidates"] = fire_stats["final_retained_fireline_units"] / fire_stats["candidate_fireline_units"]
            results["selection_by_observation_coverage"] = pd.concat([
                _selection_stage_table(candidate, ["observation_count_bin"], "coverage", stages),
                _selection_stage_table(candidate, ["recorded_outcome", "observation_count_bin"], "outcome_coverage", stages),
                _selection_stage_table(candidate, ["Year", "recorded_outcome", "observation_count_bin"], "year_outcome_coverage", stages),
            ], ignore_index=True)
        else:
            results["selection_by_observation_coverage"] = pd.DataFrame()
        results["selection_candidate_coverage_by_fire"] = fire_stats.merge(coverage, on="_irwin_norm", validate="one_to_one")
    else:
        results["selection_candidate_coverage_by_fire"] = pd.DataFrame()
        results["selection_by_observation_coverage"] = pd.DataFrame()

    per_unit = pd.concat(per_unit_rows, ignore_index=True).sort_values("unit_uid").reset_index(drop=True)
    assert len(per_unit) == len(final) and per_unit["unit_uid"].is_unique
    assert set(per_unit["unit_uid"]) == set(final["unit_uid"])
    numeric_coverage = [
        "nirops_observation_count", "observations_before_prediction_time", "observations_after_prediction_time",
        "hours_from_prediction_time_to_last_observation", "observations_after_qualifying_interval",
        "hours_from_qualifying_interval_end_to_last_observation", "qualifying_interval_hours",
    ]
    for col in numeric_coverage:
        if col not in per_unit:
            per_unit[col] = np.nan
    results["selection_final_observation_coverage"] = per_unit
    final_fire = final.groupby(["_irwin_norm", "Year"], observed=True).agg(
        final_fireline_units=("unit_uid", "size"), final_held=("y", "sum"),
        final_source_records=("source_record_key", "nunique")).reset_index()
    final_fire["final_burned_over"] = final_fire["final_fireline_units"] - final_fire["final_held"]
    final_fire["final_unit_held_fraction"] = final_fire["final_held"] / final_fire["final_fireline_units"]
    results["selection_final_coverage_by_fire"] = final_fire.merge(coverage, on="_irwin_norm", how="left", validate="one_to_one")

    held_rows = []
    count_metrics = ["nirops_observation_count", "observations_before_prediction_time",
                     "observations_after_prediction_time", "observations_after_qualifying_interval"]
    for metric in count_metrics:
        temp = per_unit[["Year", "_irwin_norm", "y", metric]].copy()
        temp["coverage_bin"] = _selection_count_bin(temp[metric])
        for scope, keys in [("all_years", ["coverage_bin"]), ("year", ["Year", "coverage_bin"])]:
            for key, group in temp.groupby(keys, dropna=False, observed=True, sort=True):
                key = key if isinstance(key, tuple) else (key,)
                held_rows.append({"scope": scope, "coverage_metric": metric, **dict(zip(keys, key)),
                                  "fireline_units": len(group), "fires": group["_irwin_norm"].nunique(),
                                  "held": int(group["y"].sum()), "burned_over": int((1 - group["y"]).sum()),
                                  "unit_held_fraction": group["y"].mean(),
                                  "population": "final included cohort only; not a selection-rate denominator"})
    results["selection_final_held_by_coverage"] = pd.DataFrame(held_rows)

    summary_rows = []
    for scope, keys in [("outcome", ["recorded_outcome"]), ("year_outcome", ["Year", "recorded_outcome"])]:
        for key, group in per_unit.groupby(keys, dropna=False, observed=True, sort=True):
            key = key if isinstance(key, tuple) else (key,)
            for metric in numeric_coverage:
                values = pd.to_numeric(group[metric], errors="coerce").dropna()
                summary_rows.append({"scope": scope, **dict(zip(keys, key)), "metric": metric,
                                     "fireline_units": len(group), "fires": group["_irwin_norm"].nunique(),
                                     "nonmissing": len(values), "missing": len(group)-len(values),
                                     "minimum": values.min(), "q25": values.quantile(.25),
                                     "median": values.median(), "q75": values.quantile(.75), "maximum": values.max()})
    results["selection_final_coverage_summary"] = pd.DataFrame(summary_rows)
    note("excluded_unit_prediction_time", "not_defined", "Pre/post-prediction coverage is reported only for retained fireline units; assigning artificial prediction times to rejected candidates would change the study design.")
    note("full_source_archive_selection", "not_estimated", "Not Engaged records and fires excluded before corrected-unit production have no equivalent 250 m denominator here. Do not divide final fireline-unit counts by source-record counts.")
    note("selection_bias_interpretation", "descriptive_only", "Observation coverage is operationally selected. Extra maps can supply Held follow-up or reveal conflict; coverage associations and ablation do not identify a causal selection effect.")
    if supported is None:
        note("qualifying_interval_followup", "unavailable", "Missing frozen timing metadata prevents validation of the earliest supporting interval; generic pre/post-prediction counts remain available.")
    results["selection_diagnostic_status"] = pd.DataFrame(status)
    results["selection_method_notes"] = pd.DataFrame([
        {"item": "denominator", "definition": "572200 saved corrected Held/Burned Over fireline units from 415 production fires, before 03N support/conflict filtering; the upstream production population is not the entire NIFC archive."},
        {"item": "stages", "definition": "Saved memberships: 317495 after support/no-Held-conflict rules; 315035 timing-eligible; 304143 in the final modeling dataset. No outcomes or evidence trajectories are recomputed."},
        {"item": "timestamps", "definition": "All available per-fire NIROPS timestamps are read from 03L trajectory caches. The times_ns key stores magnitude-decoded integers (often microseconds); cache names alone do not specify the unit."},
        {"item": "coverage", "definition": "Before and after exclude the prediction timestamp; through includes it. Counts include every cached map, not only the at-most-four maps used for approach predictors. Next/previous 48-hour windows include their outer boundary."},
        {"item": "followup", "definition": "For retained fireline units, the earliest supported start index and corrected timestamp are checked against prediction time. Follow-up after the qualifying interval counts maps strictly after its second map; the follow-up gap need not be at most 48 hours."},
        {"item": "asymmetry", "definition": "Burned Over support requires >50 m outside to >50 m inside between consecutive maps no more than 48 h apart. Held requires outside-to-near support, follow-up and absence of the ordered conflict evidence. These checks select observed patterns, not independently measured physical spread-speed classes."},
        {"item": "aggregation", "definition": "Held fractions and retention fractions use counts of fireline units in the stated stratum. Fire counts disclose clustering; rows and observation bins are descriptive, with no causal or independent-validation interpretation."},
        {"item": "source_notebooks", "definition": "03L_corrected_local_unit_production.ipynb; 03M_unit_evidence_rule_audit.ipynb; 03N_freeze_corrected_ml_cohort.ipynb; 04A_v2_dynamic_approach_features.ipynb."},
    ])
    return results

## 3. Model definitions

- **Simple local logistic:** relative humidity, log-transformed closest mapped fire-area distance, and recorded fireline type. The transformation permits a nonlinear distance response while retaining a small model. Regularization is fixed; imputation, scaling, and category encoding are fitted only on training fires.
- **Local nine-predictor LightGBM:** the nine numerical environmental and approach predictors from series 08, without recorded type.
- **Shared fire/day LightGBM:** the same nine quantities and the same tree settings, averaged over retained fireline units from the same fire and UTC day whose prediction times are at or before the current time. All fireline units at one exact time are processed together. No outcome is used in aggregation and future times are excluded.

The local/shared pair tests the contribution of local variation using matched predictor families and model capacity. The shared summaries still depend on the retrospectively selected fireline units; they are not a validated live-fire selection procedure. Within-day summaries can change as prediction time advances. They do not assert that weather is spatially uniform or that all model skill can be partitioned into within-fire and between-fire effects.

The saved full LightGBM predictions provide the original 803-predictor reference. Recorded fireline-type availability at the historical prediction time remains unverified.

In [ ]:
"""08E diagnostic models. Embedded verbatim in the self-contained notebook.

No production fitting happens on import. Call build_diagnostic_features once,
then pass whole-fire-disjoint training/test subsets to fit_predict_diagnostic.
"""

import inspect
import time
import warnings

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits


DIAGNOSTIC_NUMERIC_COLUMNS = (
    "x_humidity", "x_gust", "x_wind", "x_temperature", "x_slope",
    "x_elevation", "x_crown_depth", "x_canopy_cover", "x_distance",
)
DIAGNOSTIC_COARSE_COLUMNS = tuple("coarse_" + c for c in DIAGNOSTIC_NUMERIC_COLUMNS)
DIAGNOSTIC_MODEL_NAMES = (
    "SIMPLE_LOCAL_LOGISTIC", "LOCAL_NINE_LIGHTGBM", "COARSE_FIRE_DAY_LIGHTGBM",
)

# Fixed capacity copied from the saved 05H/05C contract. Only random_state and
# CPU n_jobs are supplied by this diagnostic run. No tuning/early stopping.
# The archived subsample=.85 is retained literally; with LightGBM's default
# subsample_freq=0 it does not enable row bagging.
DIAGNOSTIC_LGBM_PARAMETERS = {
    "objective": "binary", "n_estimators": 300, "learning_rate": 0.05,
    "num_leaves": 31, "max_depth": -1, "min_child_samples": 150,
    "subsample": 0.85, "colsample_bytree": 0.8, "reg_alpha": 0.0,
    "reg_lambda": 1.0, "max_bin": 127, "verbosity": -1,
}
DIAGNOSTIC_PARAMETER_SOURCE = (
    "outputs/forward_temporal_generalization_v1/"
    "FORWARD_TEMPORAL_MODEL_CONTRACT_V1.json"
)
DIAGNOSTIC_MISSING_TYPE = "__08E_MISSING_TYPE__"


def _diagnostic_require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def _diagnostic_identity_check(frame, description):
    _diagnostic_require(isinstance(frame, pd.DataFrame), description + " must be a DataFrame")
    for column in ("unit_uid", "_irwin_norm"):
        _diagnostic_require(column in frame, description + " lacks " + column)
        _diagnostic_require(frame[column].notna().all(), description + " has missing " + column)
    _diagnostic_require(not frame["unit_uid"].duplicated().any(), description + " has duplicate unit_uid")


def _diagnostic_numeric(frame, columns):
    missing = sorted(set(columns) - set(frame.columns))
    _diagnostic_require(not missing, "Missing diagnostic predictors: " + str(missing))
    result = pd.DataFrame(index=frame.index)
    for column in columns:
        result[column] = pd.to_numeric(frame[column], errors="coerce").astype(float)
    return result.replace([np.inf, -np.inf], np.nan)


def build_diagnostic_features(units):
    """Return a copy with log-distance and nine fire/day/as-of-anchor means.

    Each coarse value is the finite-value arithmetic mean of its predictor over
    fireline units from the SAME FIRE and UTC DATE whose saved prediction anchor
    is <= the current exact anchor. All simultaneous rows enter together. The
    operation uses only identifiers, timestamps, and the nine numeric predictors;
    it does not inspect outcomes, type, source keys, fold assignments, or years.
    Neither input rows nor their order/index are changed.

    This is an outcome-free, as-of-anchor diagnostic of the selected historical
    population, not proof those selected units were available during a live fire.
    """
    _diagnostic_identity_check(units, "Diagnostic input")
    _diagnostic_require("prediction_anchor_utc" in units, "Missing prediction_anchor_utc")
    _diagnostic_require(len(units) > 0, "Cannot build features for an empty population")
    _diagnostic_require(
        not pd.api.types.is_numeric_dtype(units["prediction_anchor_utc"].dtype),
        "Prediction anchors must be timestamps, not unqualified epoch integers",
    )
    anchors = pd.to_datetime(units["prediction_anchor_utc"], utc=True, errors="raise")
    _diagnostic_require(anchors.notna().all(), "Prediction anchors contain missing timestamps")
    numeric = _diagnostic_numeric(units, DIAGNOSTIC_NUMERIC_COLUMNS).reset_index(drop=True)
    _diagnostic_require(not numeric["x_distance"].lt(0).any(), "Negative mapped distance is invalid")
    keys = pd.DataFrame({
        "_fire": units["_irwin_norm"].astype(str).to_numpy(),
        "_day": anchors.dt.floor("D").to_numpy(),
        "_anchor": anchors.to_numpy(),
    })
    work = pd.concat([keys, numeric], axis=1)
    group_keys = ["_fire", "_day", "_anchor"]
    grouped = work.groupby(group_keys, sort=True, observed=True)
    # Aggregate simultaneous rows before the cumulative operation: no arbitrary
    # ordering within an anchor can affect features for those rows.
    sums = grouped[list(DIAGNOSTIC_NUMERIC_COLUMNS)].sum().sort_index()
    counts = grouped[list(DIAGNOSTIC_NUMERIC_COLUMNS)].count().sort_index()
    running_sums = sums.groupby(level=[0, 1], sort=False).cumsum()
    running_counts = counts.groupby(level=[0, 1], sort=False).cumsum()
    means = running_sums.div(running_counts.where(running_counts > 0))
    running_units = grouped.size().sort_index().groupby(level=[0, 1], sort=False).cumsum()
    row_keys = pd.MultiIndex.from_frame(keys[group_keys])
    aligned = means.reindex(row_keys)
    result = units.copy()
    for local, coarse in zip(DIAGNOSTIC_NUMERIC_COLUMNS, DIAGNOSTIC_COARSE_COLUMNS):
        result[coarse] = aligned[local].to_numpy(dtype=float)
    result["coarse_n_units_available"] = running_units.reindex(row_keys).to_numpy(dtype=np.int64)
    result["log1p_distance"] = np.log1p(numeric["x_distance"].to_numpy(dtype=float))
    return result


def _diagnostic_model_input(frame, model_name):
    if model_name == "SIMPLE_LOCAL_LOGISTIC":
        _diagnostic_require("type_value" in frame, "Logistic baseline requires type_value")
        design = _diagnostic_numeric(frame, ("x_distance", "x_humidity"))
        _diagnostic_require(not design["x_distance"].lt(0).any(), "Negative mapped distance is invalid")
        design["log1p_distance"] = np.log1p(design.pop("x_distance"))
        design = design[["log1p_distance", "x_humidity"]]
        # Fixed missing token is not learned from either population. The one-hot
        # category vocabulary is still fitted ONLY on the training subset.
        design["type_value"] = frame["type_value"].astype("string").fillna(
            DIAGNOSTIC_MISSING_TYPE
        ).astype(object)
        return design, ["log1p_distance", "x_humidity"], ["type_value"]
    if model_name == "LOCAL_NINE_LIGHTGBM":
        columns = list(DIAGNOSTIC_NUMERIC_COLUMNS)
    elif model_name == "COARSE_FIRE_DAY_LIGHTGBM":
        columns = list(DIAGNOSTIC_COARSE_COLUMNS)
    else:
        raise ValueError("Unknown diagnostic model: " + str(model_name))
    return _diagnostic_numeric(frame, columns), columns, []


def _diagnostic_logistic(seed):
    kwargs = {
        "C": 1.0, "solver": "lbfgs", "tol": 1e-6,
        "max_iter": 2000, "class_weight": None, "random_state": int(seed),
    }
    # sklearn 1.9 deprecates penalty in favor of l1_ratio; older versions use
    # penalty='l2'. Both branches express exactly the same fixed L2 model.
    penalty = inspect.signature(LogisticRegression).parameters.get("penalty")
    if penalty is not None and penalty.default == "deprecated":
        kwargs["l1_ratio"] = 0.0
    else:
        kwargs["penalty"] = "l2"
    return LogisticRegression(**kwargs)


def fit_predict_diagnostic(train, test, model_name, n_jobs=8, seed=20261105):
    """Fit a fixed diagnostic and return (Held probabilities, Pipeline, log).

    Call build_diagnostic_features before splitting if using the coarse model.
    Training and test fires must be disjoint. All training fireline units have
    equal weight. Imputation, scaling, and category vocabularies fit training
    rows only. Empty numeric training columns are retained and imputed to zero.
    Test outcomes are never read. No calibration, tuning, or cutoff is fitted.

    The returned native sklearn Pipeline is joblib-serializable. Its prediction
    input is the prepared numeric/type frame described by feature_columns in
    the log (use _diagnostic_model_input to prepare another diagnostic frame).
    """
    _diagnostic_identity_check(train, "Training data")
    _diagnostic_identity_check(test, "Test data")
    _diagnostic_require(len(train) > 0 and len(test) > 0, "Training and test data must be nonempty")
    _diagnostic_require(
        not set(train["_irwin_norm"].astype(str)) & set(test["_irwin_norm"].astype(str)),
        "Training and test sets overlap in fire identifiers",
    )
    _diagnostic_require(
        not set(train["unit_uid"].astype(str)) & set(test["unit_uid"].astype(str)),
        "Training and test sets overlap in fireline-unit identifiers",
    )
    _diagnostic_require("y" in train, "Training data lacks y")
    y = pd.to_numeric(train["y"], errors="coerce")
    _diagnostic_require(y.notna().all() and y.isin([0, 1]).all(), "Training y must contain only 0 and 1")
    _diagnostic_require(y.nunique() == 2, "Both outcomes are required to fit these diagnostic classifiers")
    _diagnostic_require(int(n_jobs) == n_jobs and int(n_jobs) >= 1, "n_jobs must be a positive integer")
    threads = int(n_jobs)
    prep_start = time.perf_counter()
    x_train, numeric_columns, categorical_columns = _diagnostic_model_input(train, model_name)
    x_test, _, _ = _diagnostic_model_input(test, model_name)
    numeric_steps = [("impute", SimpleImputer(strategy="median", keep_empty_features=True))]
    if model_name == "SIMPLE_LOCAL_LOGISTIC":
        numeric_steps.append(("scale", StandardScaler()))
    transformers = [("numeric", Pipeline(numeric_steps), numeric_columns)]
    if categorical_columns:
        transformers.append(("type", OneHotEncoder(handle_unknown="ignore", sparse_output=True),
                             categorical_columns))
    preprocess = ColumnTransformer(transformers, remainder="drop", sparse_threshold=0.3)
    if model_name == "SIMPLE_LOCAL_LOGISTIC":
        estimator = _diagnostic_logistic(seed)
    else:
        estimator = LGBMClassifier(**DIAGNOSTIC_LGBM_PARAMETERS,
                                   n_jobs=threads, random_state=int(seed))
    pipeline = Pipeline([("preprocess", preprocess), ("model", estimator)])
    prep_seconds = time.perf_counter() - prep_start
    fit_start = time.perf_counter()
    # Capture for the audit log, then re-emit ALL warnings. Convergence problems
    # are neither hidden nor converted into an unreported fallback model.
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        with threadpool_limits(limits=threads):
            pipeline.fit(x_train, y.to_numpy(dtype=np.int8))
    fit_seconds = time.perf_counter() - fit_start
    for item in captured:
        warnings.warn(str(item.message), item.category, stacklevel=2)
    predict_start = time.perf_counter()
    with threadpool_limits(limits=threads):
        positive_index = int(np.flatnonzero(pipeline.classes_ == 1)[0])
        probabilities = pipeline.predict_proba(x_test)[:, positive_index]
    predict_seconds = time.perf_counter() - predict_start
    _diagnostic_require(
        len(probabilities) == len(test) and np.isfinite(probabilities).all()
        and ((probabilities >= 0) & (probabilities <= 1)).all(),
        "Model produced invalid Held probabilities",
    )
    fitted_numeric = pipeline.named_steps["preprocess"].named_transformers_["numeric"]
    statistics = fitted_numeric.named_steps["impute"].statistics_
    n_iter = getattr(estimator, "n_iter_", None)
    n_iter_list = None if n_iter is None else np.atleast_1d(n_iter).astype(int).tolist()
    convergence_warning = any(issubclass(item.category, ConvergenceWarning) for item in captured)
    iteration_limit_reached = bool(
        model_name == "SIMPLE_LOCAL_LOGISTIC" and n_iter_list
        and max(n_iter_list) >= estimator.max_iter
    )
    unseen_types = 0
    if categorical_columns:
        known = set(pipeline.named_steps["preprocess"].named_transformers_["type"].categories_[0])
        unseen_types = int((~x_test["type_value"].isin(known)).sum())
    log = {
        "model_name": model_name, "training_rows": len(train), "test_rows": len(test),
        "training_fires": int(train["_irwin_norm"].nunique()),
        "test_fires": int(test["_irwin_norm"].nunique()),
        "feature_columns": list(x_train.columns), "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,
        "n_transformed_features": len(preprocess.get_feature_names_out()),
        "train_all_missing_numeric_columns": [c for c in numeric_columns if x_train[c].isna().all()],
        "training_numeric_imputation_statistics": dict(zip(numeric_columns, map(float, statistics))),
        "test_unseen_type_rows": unseen_types,
        "fit_warning_messages": [str(item.message) for item in captured],
        "convergence_warning": convergence_warning,
        "iteration_limit_reached": iteration_limit_reached, "n_iter": n_iter_list,
        "fit_convergence_review_required": convergence_warning or iteration_limit_reached,
        "prep_seconds": prep_seconds, "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds, "n_jobs": threads, "seed": int(seed),
        "training_weighting": "equal fireline units; no class balancing",
        "model_parameters": estimator.get_params(deep=False),
        "parameter_source": DIAGNOSTIC_PARAMETER_SOURCE if model_name != "SIMPLE_LOCAL_LOGISTIC" else
                            "fixed C=1 L2 logistic; no hyperparameter selection",
        "calibration": "none", "threshold_selection": "none",
    }
    return probabilities, pipeline, log


def run_model_helper_synthetic_tests():
    """Small non-production checks; returns pass flags and raises on failure."""
    # Aggregation fixture: ties, future observations, a different fire, and a
    # midnight reset. Feature values intentionally repeat across numeric fields.
    small = pd.DataFrame({
        "unit_uid": ["a1", "a2", "a3", "a4", "b1"],
        "_irwin_norm": ["a", "a", "a", "a", "b"],
        "prediction_anchor_utc": pd.to_datetime([
            "2020-01-01T01:00Z", "2020-01-01T01:00Z", "2020-01-01T02:00Z",
            "2020-01-02T00:00Z", "2020-01-01T01:00Z"], utc=True),
        "y": [0, 1, 0, 1, 0], "type_value": ["A"] * 5,
    })
    for column in DIAGNOSTIC_NUMERIC_COLUMNS:
        small[column] = [10., 20., 30., 5., 100.]
    before = build_diagnostic_features(small)
    np.testing.assert_allclose(before["coarse_x_humidity"], [15., 15., 20., 5., 100.])
    late = small.iloc[[0]].copy()
    late["unit_uid"] = "future"
    late["prediction_anchor_utc"] = pd.Timestamp("2020-01-01T23:00Z")
    late[list(DIAGNOSTIC_NUMERIC_COLUMNS)] = 1e8
    after = build_diagnostic_features(pd.concat([small, late], ignore_index=True))
    np.testing.assert_allclose(before[list(DIAGNOSTIC_COARSE_COLUMNS)],
                               after.iloc[:len(small)][list(DIAGNOSTIC_COARSE_COLUMNS)])
    changed_labels = small.copy()
    changed_labels["y"] = 1 - changed_labels["y"]
    pd.testing.assert_frame_equal(before[list(DIAGNOSTIC_COARSE_COLUMNS)],
                                  build_diagnostic_features(changed_labels)[list(DIAGNOSTIC_COARSE_COLUMNS)])
    shuffled = build_diagnostic_features(small.sample(frac=1, random_state=7)).set_index("unit_uid")
    pd.testing.assert_frame_equal(before.set_index("unit_uid")[list(DIAGNOSTIC_COARSE_COLUMNS)].sort_index(),
                                  shuffled[list(DIAGNOSTIC_COARSE_COLUMNS)].sort_index())
    all_missing = small.copy()
    all_missing["x_crown_depth"] = np.nan
    assert build_diagnostic_features(all_missing)["coarse_x_crown_depth"].isna().all()

    def synthetic_population(prefix, rows, first_year):
        index = np.arange(rows)
        frame = pd.DataFrame({
            "unit_uid": [prefix + str(i) for i in index],
            "_irwin_norm": [prefix + "fire" + str(i // 40) for i in index],
            "prediction_anchor_utc": pd.Timestamp(str(first_year) + "-01-01T00:00Z")
                                      + pd.to_timedelta((index % 40) // 10, unit="h"),
            "y": ((index % 7) < 3).astype(int),
            "type_value": np.where(index % 2, "Hand", "Dozer"),
        })
        for j, column in enumerate(DIAGNOSTIC_NUMERIC_COLUMNS):
            frame[column] = ((index * (j + 3)) % 97).astype(float) + j
        return frame

    train = synthetic_population("train", 320, 2020)
    test = synthetic_population("test", 80, 2021)
    train["x_elevation"] = np.nan
    test["x_elevation"] = 1e9
    test["x_distance"] += 1e8
    test["type_value"] = "UNSEEN_TEST_TYPE"
    train_features = build_diagnostic_features(train)
    test_features = build_diagnostic_features(test)
    passes = {
        "future_rows_do_not_change_past": True, "labels_do_not_enter_aggregation": True,
        "simultaneous_rows_share_features": True, "row_order_invariant": True,
        "day_and_fire_boundaries_respected": True, "all_missing_coarse_values_preserved": True,
    }
    for name in DIAGNOSTIC_MODEL_NAMES:
        probabilities, model, log = fit_predict_diagnostic(
            train_features, test_features, name, n_jobs=1, seed=19)
        assert probabilities.shape == (len(test),)
        if name == "SIMPLE_LOCAL_LOGISTIC":
            expected = np.nanmedian(np.log1p(train["x_distance"]))
            actual = log["training_numeric_imputation_statistics"]["log1p_distance"]
            np.testing.assert_allclose(actual, expected)
            assert log["test_unseen_type_rows"] == len(test)
            assert not log["fit_convergence_review_required"]
            passes["training_only_medians_and_categories"] = True
            # Native sklearn components survive a joblib round trip in memory.
            import io
            import joblib
            buffer = io.BytesIO()
            joblib.dump(model, buffer)
            buffer.seek(0)
            restored = joblib.load(buffer)
            prepared, _, _ = _diagnostic_model_input(test_features, name)
            np.testing.assert_allclose(restored.predict_proba(prepared)[:, 1], probabilities)
            passes["joblib_round_trip"] = True
        elif name == "LOCAL_NINE_LIGHTGBM":
            assert log["training_numeric_imputation_statistics"]["x_elevation"] == 0
            assert log["n_transformed_features"] == 9
            passes["all_missing_training_columns_retained"] = True
        else:
            assert log["n_transformed_features"] == 9
            check = test_features[["_irwin_norm", "prediction_anchor_utc"]].copy()
            check["p"] = probabilities
            assert check.groupby(["_irwin_norm", "prediction_anchor_utc"])["p"].nunique().max() == 1
            passes["coarse_predictions_constant_per_fire_timestamp"] = True
        passes[name + "_fit_predict"] = True
    return passes

## 4. Load the frozen population and create the run directory

The original fireline units, labels, prediction times, and fire partitions are checked against the master table. Only the columns needed for diagnostics are loaded from the large master file.

In [ ]:
required_paths = [UNIT_PATH, MASTER_PATH, OOF_PATH, FORWARD_PATH,
                  B_DIR/"08B_quality_checks.json", D_DIR/"08D_quality_checks.json",
                  PROJECT_ROOT/DIAGNOSTIC_PARAMETER_SOURCE]
if (B_DIR/"mapping_date_availability.csv").is_file():
    required_paths.append(B_DIR/"mapping_date_availability.csv")
required_paths += [Path(p) for p in selection_input_paths(PROJECT_ROOT)]
required_paths = sorted(set(p.resolve() for p in required_paths), key=str)
for path in required_paths: require(path.is_file(), "Missing input: " + str(path))
b_quality = json.loads((B_DIR/"08B_quality_checks.json").read_text())
d_quality = json.loads((D_DIR/"08D_quality_checks.json").read_text())
require(not b_quality["is_smoke_test"] and not d_quality["is_smoke_test"], "Production 08B and 08D inputs are required.")
for k in ["frozen_units_and_prediction_times_preserved", "all_source_joins_match"]:
    require(b_quality[k], "08B failed " + k)
require(d_quality["all_evaluation_populations_matched"], "08D population matching failed.")
contract=json.loads((PROJECT_ROOT/DIAGNOSTIC_PARAMETER_SOURCE).read_text())
require(all(contract["parameters"][k]==v for k,v in DIAGNOSTIC_LGBM_PARAMETERS.items()), "Diagnostic tree settings differ from the pinned primary capacity.")

units = pd.read_parquet(UNIT_PATH).sort_values("unit_uid").reset_index(drop=True)
require(len(units)==304143 and units["_irwin_norm"].nunique()==381, "Unexpected frozen population.")
require(units["unit_uid"].notna().all() and units["unit_uid"].is_unique, "Invalid unit IDs.")
require(units["source_record_key"].nunique()==142999, "Unexpected source-record membership.")
require(units["y"].isin([0,1]).all() and units["y"].eq(units["hold_label"]).all(), "Outcome mismatch.")
require(int(units["y"].sum())==205926, "Held count changed.")
for k in ["Year", "outer_fold"]:
    require(units.groupby("_irwin_norm")[k].nunique().eq(1).all(), "Fire spans " + k)
require(set(units["outer_fold"])==set(range(5)), "Expected the saved five folds.")
identity_columns = ["unit_uid", "_irwin_norm", "source_record_key", "Year", "outer_fold", "y", "prediction_anchor_utc"]
identity_hash = frame_hash(units, identity_columns)
master = pd.read_parquet(MASTER_PATH, columns=["unit_uid", "hold_label", "prediction_anchor_utc", "IncidentName", "longitude", "latitude"])
checked = units[["unit_uid", "y", "prediction_anchor_utc"]].merge(master, on="unit_uid", validate="one_to_one", suffixes=("_08B","_master"))
require(checked["y"].eq(checked["hold_label"]).all(), "Master labels differ.")
require(pd.to_datetime(checked["prediction_anchor_utc_08B"], utc=True).eq(pd.to_datetime(checked["prediction_anchor_utc_master"], utc=True)).all(), "Master prediction times differ.")
units = units.merge(master[["unit_uid", "IncidentName", "longitude", "latitude"]], on="unit_uid", validate="one_to_one")
units["prediction_anchor_utc"] = pd.to_datetime(units["prediction_anchor_utc"], utc=True)
require(units["prediction_anchor_utc"].notna().all(), "Missing prediction times.")
raw_fields = {
    "humidity": "hrrr_window_0_18h_rh2_pct_mean", "gust": "hrrr_gust_mps_p18h",
    "wind": "hrrr_window_0_18h_wind10_speed_mean_mps", "temperature": "hrrr_window_0_18h_t2_c_max",
    "slope": "terrain_slope_mean_deg", "elevation": "terrain_elevation_mean_m",
    "crown_depth": "landfire_line_crown_depth_mean", "canopy_cover": "landfire_line_cc_mean",
    "distance": "local_fire_to_line_distance_geom_m"}
inventory = []
for key, field in raw_fields.items():
    x = pd.to_numeric(units[field], errors="coerce").replace([np.inf,-np.inf], np.nan)
    valid = x.notna()
    if key in ["humidity", "canopy_cover"]: valid &= x.between(0,100)
    if key=="slope": valid &= x.between(0,90)
    if key in ["gust", "wind", "crown_depth", "distance"]: valid &= x.ge(0)
    units["x_"+key] = x.where(valid)
    inventory.append({"predictor": "x_"+key, "source_field": field, "missing_or_invalid": int((~valid).sum())})
units["type_value"] = units["fireline_type"].fillna("__MISSING__").astype(str)
units = build_diagnostic_features(units)
tables["predictor_inventory"] = pd.DataFrame(inventory)

print("Fingerprinting", len(required_paths), "input files ...", flush=True)
input_manifest = {}
for path in required_paths:
    stat = path.stat()
    input_manifest[str(path.relative_to(PROJECT_ROOT))] = {"bytes": stat.st_size, "modified_ns": stat.st_mtime_ns, "sha256": file_hash(path)}
versions = {name: package_version(name) for name in ["numpy","pandas","pyarrow","scikit-learn","lightgbm","matplotlib","joblib"]}
run_spec = {"build": BUILD_ID, "source_sha256": source_hash(), "seed": RANDOM_SEED,
            "is_smoke_test": IS_SMOKE_TEST, "smoke_units_per_fire": SMOKE_UNITS_PER_FIRE,
            "bootstrap_replicates": BOOTSTRAP_REPLICATES, "n_jobs": N_JOBS,
            "packages": versions, "inputs": input_manifest,
            "procedures": ["SIMPLE_LOCAL_LOGISTIC", "LOCAL_NINE_LIGHTGBM", "COARSE_FIRE_DAY_LIGHTGBM"],
            "training_weights": "equal fireline units", "evaluation_weights": ["unit", "fire"],
            "coarse_definition": "cumulative finite means within fire and UTC day, exact anchor ties updated together",
            "new_tuning_calibration_threshold_selection": False}
RUN_FINGERPRINT = hashlib.sha256(json.dumps(run_spec, sort_keys=True).encode()).hexdigest()
if RESUME_RUN_DIR:
    RUN_DIR = Path(RESUME_RUN_DIR).resolve()
    require(RUN_DIR.is_relative_to((PROJECT_ROOT/"outputs/08_condition_outcomes").resolve()), "Resume directory must be under outputs/08_condition_outcomes.")
    prior_plan = json.loads((RUN_DIR/"08E_fixed_run_plan.json").read_text())
    require(prior_plan["fingerprint"]==RUN_FINGERPRINT, "Resume fingerprint differs. Start a new run; do not mix checkpoints.")
else:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid.uuid4().hex[:6]
    RUN_DIR = PROJECT_ROOT/"outputs/08_condition_outcomes"/("08E_SMOKE_"+stamp if IS_SMOKE_TEST else "08E_"+stamp)
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    write_json(RUN_DIR/"08E_fixed_run_plan.json", {"fingerprint": RUN_FINGERPRINT, "specification": run_spec})
MODEL_DIR = RUN_DIR/"models"; MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR = RUN_DIR/"figures"; FIG_DIR.mkdir(exist_ok=True)

if IS_SMOKE_TEST:
    population = units.groupby("_irwin_norm", group_keys=False).sample(frac=1, random_state=RANDOM_SEED).groupby("_irwin_norm", group_keys=False).head(SMOKE_UNITS_PER_FIRE).sort_values("unit_uid").copy()
else:
    population = units.copy()
require(population["unit_uid"].is_unique, "Duplicate model rows.")
phase_done.add("load")
print(f"Verified {len(units):,} fireline units / 381 fires; model population {len(population):,}.")
print("SAVE THIS PATH TO RESUME:", RUN_DIR)

## 5. Selection, observation coverage, and mapped-date audits

The cumulative stage counts use the saved rule memberships. Observation-count associations are descriptive: more observations can supply approach/follow-up evidence and can also reveal a contradiction. Differences do not prove that observation coverage caused an outcome. Missing upstream records are reported as unavailable instead of silently substituting the final population.

In [ ]:
need("load")
selection_tables = run_selection_diagnostics(PROJECT_ROOT, units)
require(not set(selection_tables).intersection(tables), "Duplicate output table names.")
tables.update(selection_tables)
for name, table in selection_tables.items():
    table.to_csv(RUN_DIR/(name+".csv"), index=False)
display(tables["selection_diagnostic_status"])
if not tables["selection_stage_counts"].empty:
    display(tables["selection_stage_counts"].loc[lambda d:d["scope"].eq("overall"),
            ["stage","retained_fireline_units","retained_fires","retained_unit_held_fraction"]])
display(tables["selection_final_coverage_summary"].loc[lambda d:d["scope"].eq("outcome"),
        ["recorded_outcome","metric","minimum","median","maximum"]])
date_path = B_DIR/"mapping_date_availability.csv"
if date_path.is_file():
    tables["prior_mapping_date_audit"] = pd.read_csv(date_path)
    display(tables["prior_mapping_date_audit"])
phase_done.add("selection")

## 6. Population description and concentration

The top-ten share is the fraction of retained fireline units contributed by the ten fires with the most fireline units in the stated population. The geographic figure uses the mean coordinates of each fire's retained fireline units, not ignition locations. Geography and fire size are descriptive diagnostics, not new blocked validation experiments.

In [ ]:
need("load")
def population_summary(frame, keys):
    out = frame.groupby(keys, dropna=False, observed=True).agg(
        fireline_units=("unit_uid","size"), fires=("_irwin_norm","nunique"),
        source_records=("source_record_key","nunique"), held=("y","sum")).reset_index()
    out["burned_over"] = out["fireline_units"]-out["held"]
    out["observed_held_fraction"] = out["held"]/out["fireline_units"]
    return out

tables["population_by_year"] = population_summary(units, ["Year"])
tables["population_by_type"] = population_summary(units, ["type_value"])
tables["population_by_year_type"] = population_summary(units, ["Year","type_value"])
fire_sizes = units.groupby(["_irwin_norm","Year"], observed=True).agg(
    IncidentName=("IncidentName","first"), fireline_units=("unit_uid","size"),
    source_records=("source_record_key","nunique"), held=("y","sum"),
    longitude=("longitude","mean"), latitude=("latitude","mean")).reset_index()
fire_sizes["observed_held_fraction"] = fire_sizes["held"]/fire_sizes["fireline_units"]
concentration = []; cumulative = []
for scope, sub in [("all", fire_sizes)] + [(str(y), f) for y,f in fire_sizes.groupby("Year")]:
    ordered = sub.sort_values(["fireline_units","_irwin_norm"], ascending=[False,True]).copy()
    ordered["scope"] = scope; ordered["rank"] = np.arange(1,len(ordered)+1)
    ordered["unit_share"] = ordered["fireline_units"]/ordered["fireline_units"].sum()
    ordered["cumulative_unit_share"] = ordered["unit_share"].cumsum()
    cumulative.append(ordered)
    concentration.append({"scope":scope, "fires":len(ordered), "fireline_units":int(ordered["fireline_units"].sum()),
                          "largest_fire_share":ordered["unit_share"].iloc[0],
                          "top_10_unit_share":ordered["unit_share"].head(10).sum(),
                          "largest_fire":ordered["IncidentName"].iloc[0]})
tables["fire_size_concentration"] = pd.DataFrame(concentration)
tables["ranked_fire_sizes"] = pd.concat(cumulative, ignore_index=True)
tables["fire_locations_and_sizes"] = fire_sizes
display(tables["population_by_year"])
display(tables["fire_size_concentration"])
phase_done.add("description")

## 7. Statistical helpers and software safeguards

Every split has two training-only constants: the Held fraction across training fireline units, and the mean of training fires' Held fractions. The unit-weighted evaluation uses the first; the equal-fire evaluation uses the second. Test outcomes never determine either constant.

Bootstrap intervals resample complete fires, retaining duplicate draws. Predictions, model settings, and grouping definitions stay fixed. Intervals are pointwise 95% intervals and are not adjusted for the number of comparisons. ROC-AUC and AP are reported as point estimates; the paired intervals below concern Brier score and log loss.

In [ ]:
def historical_priors(frame):
    require(len(frame)>0 and frame["y"].isin([0,1]).all(), "Invalid training outcomes.")
    return float(frame["y"].mean()), float(frame.groupby("_irwin_norm")["y"].mean().mean())

def assert_split(train, test, chronological=False):
    require(len(train)>0 and len(test)>0, "Empty split.")
    require(set(train["_irwin_norm"]).isdisjoint(test["_irwin_norm"]), "Training/test fire overlap.")
    require(set(train["source_record_key"]).isdisjoint(test["source_record_key"]), "Training/test source overlap.")
    require(set(train["unit_uid"]).isdisjoint(test["unit_uid"]), "Training/test unit overlap.")
    if chronological: require(train["Year"].max()<test["Year"].min(), "Forward-year temporal overlap.")

def losses(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float)
    require(np.isfinite(p).all() and ((p>=0)&(p<=1)).all(), "Invalid probabilities.")
    q = np.clip(p, 1e-15, 1-1e-15)
    return {"brier":(p-y)**2, "log_loss":-(y*np.log(q)+(1-y)*np.log1p(-q))}

def resampled_average(sums, counts, draws, weighting):
    sums = np.asarray(sums, float); counts = np.asarray(counts, float)
    if weighting=="unit": return (draws@sums)/(draws@counts)
    if weighting=="fire": return (draws@(sums/counts))/draws.sum(axis=1)
    raise ValueError(weighting)

def align_checkpoint(frame, test):
    require(frame["unit_uid"].is_unique and set(frame["unit_uid"])==set(test["unit_uid"]), "Checkpoint test IDs differ.")
    frame = frame.set_index("unit_uid").loc[test["unit_uid"]].reset_index()
    require(np.array_equal(frame["y"], test["y"]), "Checkpoint outcomes differ.")
    require(np.array_equal(frame["_irwin_norm"], test["_irwin_norm"]), "Checkpoint fires differ.")
    require(np.isfinite(frame["p"]).all() and frame["p"].between(0,1).all(), "Invalid checkpoint probabilities.")
    return frame["p"].to_numpy()

# Fire-average prior is unchanged by duplicating all rows of one fire.
toy = pd.DataFrame({"_irwin_norm":["A","B","B"], "y":[1,0,0]})
toy2 = pd.concat([toy, toy.loc[toy["_irwin_norm"].eq("A")]], ignore_index=True)
require(historical_priors(toy)[1]==historical_priors(toy2)[1] and historical_priors(toy)[0]!=historical_priors(toy2)[0], "Prior-weight test failed.")
# A,A,B: keep both copies of A. Explicit pooled=(.2+.2+1.8)/(1+1+2).
draw = np.array([[2,1]])
require(np.isclose(resampled_average([.2,1.8],[1,2],draw,"unit")[0], .55), "Duplicate-fire pooled draw failed.")
require(np.isclose(resampled_average([.2,1.8],[1,2],draw,"fire")[0], (2*.2+.9)/3), "Duplicate-fire equal-fire draw failed.")
dummy_test = pd.DataFrame({"unit_uid":["U1","U2"], "y":[1,0], "_irwin_norm":["A","B"]})
dummy_cp = dummy_test.assign(p=[.8,.3]).iloc[::-1]
require(np.allclose(align_checkpoint(dummy_cp,dummy_test),[.8,.3]), "ID alignment failed.")
require(frame_hash(dummy_test.assign(x=[1.,2.]),["unit_uid","y","x"])!=frame_hash(dummy_test.assign(x=[1.,3.]),["unit_uid","y","x"]), "Predictor fingerprint failed.")
fingerprint_fixture={"cells":[{"cell_type":"code","source":"RESUME_RUN_DIR = None\nRANDOM_SEED = 17"}]}
resume_fixture=copy.deepcopy(fingerprint_fixture)
resume_fixture["cells"][0]["source"]="RESUME_RUN_DIR = 'existing/run'\nRANDOM_SEED = 17"
require(notebook_source_fingerprint(fingerprint_fixture)==notebook_source_fingerprint(resume_fixture),"Resume-path edit incorrectly invalidates code fingerprint.")
resume_fixture["cells"][0]["source"]="RESUME_RUN_DIR = 'existing/run'\nRANDOM_SEED = 18"
require(notebook_source_fingerprint(fingerprint_fixture)!=notebook_source_fingerprint(resume_fixture),"Scientific-setting edit must invalidate code fingerprint.")
# Model-specific tests check exact-time ties, future exclusion, and train-only preprocessing.
if "run_model_helper_synthetic_tests" in globals():
    model_safeguards = run_model_helper_synthetic_tests()
    print(model_safeguards)
else:
    raise AssertionError("Missing model-helper synthetic checks.")
phase_done.add("safeguards")
print("Statistical and identifier safeguards passed.")

## 8. Train the three diagnostic models, with verified resume

The fixed procedures are fitted separately for each saved fold and each forward-year split. A matched pair of local/shared models has identical training rows, numerical families, tree settings, and training weights. The shared model receives only the shared summaries. Original full-model predictions are joined by fireline-unit ID.

Each fit saves predictions, the fitted pipeline/model, training priors, warnings, and timing. A completed checkpoint is committed last. Re-running this cell resumes completed fits after checking their contents. Convergence warnings remain visible and are included in the fit log.

In [ ]:
need("load", "safeguards")
model_names = ["SIMPLE_LOCAL_LOGISTIC", "LOCAL_NINE_LIGHTGBM", "COARSE_FIRE_DAY_LIGHTGBM"]
probability_columns = {"SIMPLE_LOCAL_LOGISTIC":"p_logistic", "LOCAL_NINE_LIGHTGBM":"p_local_nine", "COARSE_FIRE_DAY_LIGHTGBM":"p_shared"}
splits = [{"scheme":"five_fold", "split":str(f), "fold":f} for f in range(5)]
splits += [{"scheme":"forward_year", "split":str(y), "year":y} for y in range(2021,2025)]
oof = pd.read_parquet(OOF_PATH)
forward = pd.read_parquet(FORWARD_PATH)
require(oof["unit_uid"].is_unique and forward["unit_uid"].is_unique, "Duplicate saved prediction IDs.")
require(set(oof["unit_uid"])==set(units["unit_uid"]), "Saved five-fold population differs.")
require(set(forward["unit_uid"])==set(units.loc[units["Year"].ge(2021),"unit_uid"]), "Saved forward population differs.")
oof = oof.set_index("unit_uid"); forward = forward.set_index("unit_uid")
all_prediction_frames = []; fit_rows = []; split_rows = []
started_all = time.perf_counter()
checkpoint_feature_columns = ["unit_uid", "_irwin_norm", "source_record_key", "Year", "outer_fold", "y", "prediction_anchor_utc", "type_value", "log1p_distance"]
checkpoint_feature_columns += ["x_"+k for k in raw_fields] + ["coarse_x_"+k for k in raw_fields]
for split in splits:
    if split["scheme"]=="five_fold":
        train = population.loc[population["outer_fold"].ne(split["fold"])].copy()
        test = population.loc[population["outer_fold"].eq(split["fold"])].copy()
        ref = oof.loc[test["unit_uid"]]
        require(np.array_equal(ref["outer_fold"], test["outer_fold"]), "Saved outer folds differ.")
        p_full = ref["pred_prob"].to_numpy()
    else:
        train = population.loc[population["Year"].lt(split["year"])].copy()
        test = population.loc[population["Year"].eq(split["year"])].copy()
        ref = forward.loc[test["unit_uid"]]
        p_full = ref["pred_forward_fixed_05c"].to_numpy()
    assert_split(train, test, chronological=split["scheme"]=="forward_year")
    require(np.array_equal(ref["hold_label"],test["y"]) and np.array_equal(ref["_irwin_norm"],test["_irwin_norm"]), "Saved reference alignment failed.")
    prior_unit, prior_fire = historical_priors(train)
    if split["scheme"]=="forward_year" and not IS_SMOKE_TEST:
        require(np.allclose(ref["pred_historical_prevalence_prior"], prior_unit), "Historical unit baseline differs from saved forward analysis.")
    pred = test[["unit_uid","_irwin_norm","source_record_key","Year","outer_fold","y","prediction_anchor_utc"]].copy()
    pred["scheme"] = split["scheme"]; pred["split"] = split["split"]
    pred["p_full"] = p_full; pred["p_baseline_unit"] = prior_unit; pred["p_baseline_fire"] = prior_fire
    train_fingerprint = frame_hash(train, checkpoint_feature_columns)
    test_fingerprint = frame_hash(test, checkpoint_feature_columns)
    print(f"\n{split['scheme']} {split['split']}: {len(train):,} train / {len(test):,} test fireline units", flush=True)
    for name in model_names:
        model_id = name+"__"+split["scheme"]+"_"+split["split"]
        folder = MODEL_DIR/model_id; folder.mkdir(exist_ok=True)
        checkpoint = folder/"complete.json"
        key = {"run_fingerprint":RUN_FINGERPRINT, "model":name, "split":split, "train":train_fingerprint, "test":test_fingerprint}
        fingerprint = hashlib.sha256(json.dumps(key, sort_keys=True).encode()).hexdigest()
        if checkpoint.is_file():
            completed = json.loads(checkpoint.read_text())
            require(completed["fingerprint"]==fingerprint, "Incompatible checkpoint: "+model_id)
            for filename, expected_hash in completed["files"].items():
                require(file_hash(folder/filename)==expected_hash, "Checkpoint content changed: "+model_id+"/"+filename)
            p = align_checkpoint(pd.read_parquet(folder/"test_predictions.parquet"), test)
            log = completed["fit_log"]
            print(name+": resumed verified predictions", flush=True)
        else:
            start = time.perf_counter()
            p, fitted, info = fit_predict_diagnostic(train, test, name, n_jobs=N_JOBS, seed=RANDOM_SEED)
            losses(test["y"], p)
            log = {"model_id":model_id, "scheme":split["scheme"], "split":split["split"], "model":name,
                   "train_units":len(train), "test_units":len(test), "train_fires":train["_irwin_norm"].nunique(),
                   "test_fires":test["_irwin_norm"].nunique(), "seconds":time.perf_counter()-start,
                   "baseline_unit":prior_unit, "baseline_fire":prior_fire, **info}
            cp = test[["unit_uid","_irwin_norm","y"]].copy(); cp["p"] = p
            cp.to_parquet(folder/"test_predictions.parquet", index=False)
            joblib.dump(fitted, folder/"fitted_model.joblib", compress=3)
            write_json(folder/"fit_info.json", log)
            write_json(checkpoint, {"fingerprint":fingerprint, "key":key, "fit_log":log,
                                   "files":{f:file_hash(folder/f) for f in ["test_predictions.parquet","fitted_model.joblib","fit_info.json"]}})
            print(f"{name}: {log['seconds']:.1f}s", flush=True)
        pred[probability_columns[name]] = p
        fit_rows.append(log)
        pd.DataFrame(fit_rows).to_csv(RUN_DIR/"08E_fit_progress.csv", index=False)
    split_rows.append({**split, "train_years": ",".join(map(str,sorted(train["Year"].unique()))),
                       "train_units":len(train), "test_units":len(test), "train_fires":train["_irwin_norm"].nunique(),
                       "test_fires":test["_irwin_norm"].nunique(), "baseline_unit":prior_unit, "baseline_fire":prior_fire})
    all_prediction_frames.append(pred)
predictions = pd.concat(all_prediction_frames, ignore_index=True)
require(not predictions.duplicated(["scheme","unit_uid"]).any(), "Duplicate out-of-sample predictions.")
for scheme in ["five_fold","forward_year"]:
    expected = population if scheme=="five_fold" else population.loc[population["Year"].ge(2021)]
    require(set(predictions.loc[predictions["scheme"].eq(scheme),"unit_uid"])==set(expected["unit_uid"]), "Incomplete predictions for "+scheme)
predictions.to_parquet(RUN_DIR/"08E_matched_predictions.parquet", index=False)
tables["fit_log"] = pd.DataFrame(fit_rows)
tables["split_inventory"] = pd.DataFrame(split_rows)
phase_done.add("predictions")
print(f"All 27 fit checkpoints and matched predictions are complete. Fit-loop elapsed: {(time.perf_counter()-started_all)/60:.1f} minutes.")

## 9. Matched errors, ranking, reliability, and uncertainty

Negative **model-minus-comparator** differences favor the model named in the first column. All comparisons use identical test fireline units. The local-nine versus shared comparison is the targeted test of local variation; the full model comparison also changes the predictor set.

Reliability bins use fixed probability intervals and equal fireline-unit weights. Each bin reports its number of fireline units and fires. Sparse bins should not be interpreted as precise calibration estimates. Whole-fire bootstrap intervals are supplied for the paired errors, not for the reliability-bin means.

In [ ]:
need("predictions")
PRED_LABELS = {"p_full":"Full LightGBM (803)", "p_logistic":"Simple local logistic",
               "p_local_nine":"Local LightGBM (9)", "p_shared":"Shared fire/day LightGBM (9)",
               "p_baseline_unit":"Historical unit Held fraction", "p_baseline_fire":"Historical fire-average Held fraction"}
model_cols = ["p_full","p_logistic","p_local_nine","p_shared"]
metric_rows = []; paired_rows = []; reliability_rows = []; per_fire_rows = []
scopes = []
for scheme, frame in predictions.groupby("scheme", sort=True):
    scopes.append((scheme,"all",frame))
    scopes.extend((scheme,str(year),g) for year,g in frame.groupby("Year", sort=True))
for scope_index, (scheme,scope,frame) in enumerate(scopes):
    print("Evaluating", scheme, scope, flush=True)
    fire_ids, fire_codes = np.unique(frame["_irwin_norm"].to_numpy(), return_inverse=True)
    n_fires = len(fire_ids); n = np.bincount(fire_codes, minlength=n_fires).astype(float)
    y = frame["y"].to_numpy(float)
    group_indices = [np.flatnonzero(fire_codes==k) for k in range(n_fires)]
    rng = np.random.default_rng(RANDOM_SEED+1000+scope_index)
    draws = rng.multinomial(n_fires, np.full(n_fires,1/n_fires), size=BOOTSTRAP_REPLICATES)
    sums = {}; sampled = {}
    for col in PRED_LABELS:
        p = frame[col].to_numpy(float)
        errs = losses(y,p)
        within_auc = []; within_ap = []
        for k, idx in enumerate(group_indices):
            both = len(np.unique(y[idx]))==2
            auc = roc_auc_score(y[idx],p[idx]) if both else np.nan
            ap = average_precision_score(y[idx],p[idx]) if both else np.nan
            if both: within_auc.append(auc); within_ap.append(ap)
            per_fire_rows.append({"scheme":scheme,"scope":scope,"model":col,"fire":fire_ids[k],
                                  "fireline_units":len(idx),"held":int(y[idx].sum()),
                                  "brier":float(errs["brier"][idx].mean()),"log_loss":float(errs["log_loss"][idx].mean()),
                                  "within_fire_roc_auc":auc,"within_fire_ap":ap})
        both_all = len(np.unique(y))==2
        result = {"scheme":scheme,"scope":scope,"model":col,"fireline_units":len(frame),"fires":n_fires,
                  "fires_with_both_outcomes":len(within_auc),"observed_held_fraction":float(y.mean()),
                  "mean_predicted_held":float(p.mean()),
                  "pooled_roc_auc":roc_auc_score(y,p) if both_all else np.nan,
                  "pooled_ap":average_precision_score(y,p) if both_all else np.nan,
                  "within_fire_roc_auc":float(np.mean(within_auc)) if within_auc else np.nan,
                  "within_fire_ap":float(np.mean(within_ap)) if within_ap else np.nan}
        for metric,e in errs.items():
            values = np.bincount(fire_codes, weights=e, minlength=n_fires)
            sums[(col,metric)] = values
            result["pooled_"+metric] = float(e.mean())
            result["fire_average_"+metric] = float((values/n).mean())
            for weighting in ["unit","fire"]:
                sampled[(col,metric,weighting)] = resampled_average(values,n,draws,weighting)
        metric_rows.append(result)
        bin_id = np.minimum((p*10).astype(int),9)
        for b in range(10):
            keep = bin_id==b
            reliability_rows.append({"scheme":scheme,"scope":scope,"model":col,"bin":b,
                                     "lower":b/10,"upper":(b+1)/10,"fireline_units":int(keep.sum()),
                                     "fires":len(np.unique(fire_codes[keep])),
                                     "mean_prediction":float(p[keep].mean()) if keep.any() else np.nan,
                                     "observed_held_fraction":float(y[keep].mean()) if keep.any() else np.nan})
    for weighting in ["unit","fire"]:
        baseline = "p_baseline_unit" if weighting=="unit" else "p_baseline_fire"
        pairs = [(col,baseline) for col in model_cols] + [("p_local_nine","p_shared"),("p_full","p_local_nine"),("p_full","p_shared"),("p_full","p_logistic")]
        for a,b in pairs:
            for metric in ["brier","log_loss"]:
                a_mean = float(sums[(a,metric)].sum()/n.sum()) if weighting=="unit" else float((sums[(a,metric)]/n).mean())
                b_mean = float(sums[(b,metric)].sum()/n.sum()) if weighting=="unit" else float((sums[(b,metric)]/n).mean())
                delta = sampled[(a,metric,weighting)]-sampled[(b,metric,weighting)]
                lo,hi = np.quantile(delta,[.025,.975])
                paired_rows.append({"scheme":scheme,"scope":scope,"model":a,"comparator":b,"weighting":weighting,
                                    "metric":metric,"model_error":a_mean,"comparator_error":b_mean,"difference":a_mean-b_mean,
                                    "ci_lower":lo,"ci_upper":hi,"fires":n_fires,"fireline_units":len(frame),
                                    "bootstrap_replicates":BOOTSTRAP_REPLICATES})
tables["performance_metrics"] = pd.DataFrame(metric_rows)
tables["paired_error_comparisons"] = pd.DataFrame(paired_rows)
tables["per_fire_performance"] = pd.DataFrame(per_fire_rows)
tables["reliability_bins"] = pd.DataFrame(reliability_rows)
display(tables["performance_metrics"].query("scope == 'all'")[["scheme","model","pooled_brier","fire_average_brier","pooled_roc_auc","within_fire_roc_auc"]])
phase_done.add("evaluation")

## 10. Figures

Every figure is exported as a vector PDF and a PNG. The selection figures describe the existing historical inclusion process. They are not independent label-agreement tests. The geographic locations describe retained fireline mapping, not the complete spatial extent of fires.

In [ ]:
need("evaluation","description","selection")
def save_figure(fig, name):
    if IS_SMOKE_TEST:
        fig.text(.5,1.015,"SOFTWARE VERIFICATION — DO NOT USE AS SCIENTIFIC RESULTS", ha="center", color="darkred", fontsize=8)
    for ext in ["pdf","png"]:
        path = FIG_DIR/(name+"."+ext)
        fig.savefig(path, bbox_inches="tight")
        figure_files.append(str(path.relative_to(RUN_DIR)))
    plt.show(); plt.close(fig)

fig,ax = plt.subplots(figsize=(6.8,4.2))
for scope, group in tables["ranked_fire_sizes"].groupby("scope", sort=True):
    ax.plot(group["rank"], group["cumulative_unit_share"], label=scope, lw=2 if scope=="all" else 1)
ax.set(xlabel="Fires ordered by number of retained fireline units", ylabel="Cumulative share of fireline units", ylim=(0,1.01))
ax.yaxis.set_major_formatter(PercentFormatter(1)); ax.legend(title="Population", frameon=False, ncol=2)
ax.grid(axis="y", alpha=.2)
save_figure(fig,"08E_fire_concentration")

loc = fire_sizes.loc[fire_sizes["longitude"].between(-180,180)&fire_sizes["latitude"].between(-90,90)].copy()
require(len(loc)==len(fire_sizes), "Invalid fire coordinates; inspect before plotting coverage.")
main = loc["longitude"].between(-126,-65)&loc["latitude"].between(24,50)
parts=[(loc.loc[main],"Retained fires in the contiguous United States")]
if (~main).any(): parts.append((loc.loc[~main],"Additional retained fires"))
fig,axes = plt.subplots(1,len(parts),figsize=(7.1,4.4),squeeze=False,layout="constrained")
for ax,(part,title) in zip(axes.flat,parts):
    sc=ax.scatter(part["longitude"],part["latitude"],c=part["Year"],cmap="viridis",vmin=2020,vmax=2024,
                  s=24,edgecolors="black",linewidths=.3,alpha=.8)
    ax.set(xlabel="Longitude (degrees)",ylabel="Latitude (degrees)")
    ax.grid(alpha=.2);ax.set_title(title,fontsize=10)
fig.colorbar(sc,ax=list(axes.flat),label="Fire year",ticks=range(2020,2025),fraction=.035,pad=.03)
save_figure(fig,"08E_geographic_fire_locations")

colors = {"p_full":"#222222","p_logistic":"#0072B2","p_local_nine":"#009E73","p_shared":"#D55E00"}
fig,axes = plt.subplots(2,2,figsize=(7.1,6),sharex=True,sharey=True)
for ax,year in zip(axes.flat,range(2021,2025)):
    for col in model_cols:
        r=tables["reliability_bins"].loc[lambda d:d["scheme"].eq("forward_year")&d["scope"].eq(str(year))&d["model"].eq(col)&d["fireline_units"].gt(0)]
        ax.plot(r["mean_prediction"],r["observed_held_fraction"],"o-",ms=3,lw=1,color=colors[col],label=PRED_LABELS[col])
    ax.plot([0,1],[0,1],ls="--",lw=.7,color="0.5")
    ax.set(title=str(year),xlim=(0,1),ylim=(0,1));ax.grid(alpha=.15)
for ax in axes[-1,:]:ax.set_xlabel("Mean predicted Held probability")
for ax in axes[:,0]:ax.set_ylabel("Observed Held fraction")
handles,labels=axes[0,0].get_legend_handles_labels()
fig.legend(handles,labels,loc="lower center",bbox_to_anchor=(.5,-.02),ncol=2,frameon=False,fontsize=8)
fig.tight_layout(rect=(0,.05,1,1))
save_figure(fig,"08E_forward_reliability")

fig,axes = plt.subplots(1,2,figsize=(7.1,3.4),sharey=True)
for ax,weighting,title in zip(axes,["unit","fire"],["Equal fireline units","Equal fires"]):
    df=tables["paired_error_comparisons"]
    b="p_baseline_unit" if weighting=="unit" else "p_baseline_fire"
    df=df.loc[df["scheme"].eq("forward_year")&df["scope"].eq("all")&df["metric"].eq("brier")&df["weighting"].eq(weighting)&df["comparator"].eq(b)].set_index("model").loc[model_cols]
    for j,col in enumerate(model_cols):
        row=df.loc[col]
        ax.plot([row.ci_lower,row.ci_upper],[j,j],color=colors[col],lw=1.5)
        ax.plot(row.difference,j,"o",color=colors[col],ms=4)
    ax.axvline(0,color="0.4",ls="--",lw=.8)
    ax.set(title=title,yticks=range(4),yticklabels=[PRED_LABELS[c] for c in model_cols])
    ax.grid(axis="x",alpha=.2)
axes[0].invert_yaxis()
fig.supxlabel("Brier score difference (model minus matched baseline)",y=.02,fontsize=10)
fig.tight_layout(rect=(0,.08,1,1))
save_figure(fig,"08E_forward_baseline_comparisons")

# Full-record coverage is descriptive and never used by a fitted model.
cov=tables.get("selection_final_coverage_by_fire")
if cov is not None:
    numeric_counts=[c for c in cov if c in ["nirops_observation_count"]]
    if numeric_counts:
        id_col="_irwin_norm" if "_irwin_norm" in cov else "fire_id"
        if id_col in cov:
            d=fire_sizes.merge(cov[[id_col,numeric_counts[0]]],left_on="_irwin_norm",right_on=id_col,validate="one_to_one")
            fig,ax=plt.subplots(figsize=(5.5,3.8))
            ax.scatter(d[numeric_counts[0]],d["observed_held_fraction"],s=12+60*np.sqrt(d["fireline_units"]/d["fireline_units"].max()),alpha=.65,color="#0072B2")
            ax.set(xlabel="Available NIROPS observations in the full record",ylabel="Observed Held fraction within each fire",ylim=(0,1.02))
            ax.grid(alpha=.2);save_figure(fig,"08E_observation_coverage_and_held_fraction")

selection=tables.get("selection_by_observation_coverage",pd.DataFrame())
if not selection.empty:
    bins=["0","1-2","3-5","6-10","11-20","21-40","41+","Unavailable"]
    sub=selection.loc[selection["scope"].eq("outcome_coverage")&selection["stage"].eq("final_modeling_cohort")]
    active=[b for b in bins if b in set(sub["observation_count_bin"])]
    fig,ax=plt.subplots(figsize=(6.8,3.7))
    for outcome,color in [("Held","#0072B2"),("Burned Over","#D55E00")]:
        d=sub.loc[sub["recorded_outcome"].eq(outcome)].set_index("observation_count_bin").reindex(active)
        ax.plot(range(len(active)),d["retained_fraction_of_candidates"],"o-",label=outcome,color=color)
    ax.set(xticks=range(len(active)),xticklabels=active,xlabel="NIROPS observations in the available full record",
           ylabel="Fraction of candidate fireline units retained",ylim=(0,1))
    ax.yaxis.set_major_formatter(PercentFormatter(1));ax.legend(title="Recorded NIFC outcome",frameon=False);ax.grid(axis="y",alpha=.2)
    save_figure(fig,"08E_selection_by_observation_coverage")
phase_done.add("figures")

## 11. Save the report and verify completion

Start with `08E_FINDINGS_REPORT.md`, then inspect `paired_error_comparisons.csv`, the selection tables, and the figures. The report retains every planned model and both weighting choices. A favorable comparison is evidence about the selected historical population; it is not independent confirmation on newly collected fires.

In [ ]:
need("load","selection","description","safeguards","predictions","evaluation","figures")
require(len(units)==304143 and units["unit_uid"].is_unique, "Frozen membership changed.")
# Compare identity after normalizing timestamps against the original input, not a dtype-dependent hash.
original=pd.read_parquet(UNIT_PATH,columns=identity_columns)
original["prediction_anchor_utc"]=pd.to_datetime(original["prediction_anchor_utc"],utc=True)
pd.testing.assert_frame_equal(units[identity_columns].sort_values("unit_uid").reset_index(drop=True),original.sort_values("unit_uid").reset_index(drop=True),check_dtype=False)
require(source_hash()==run_spec["source_sha256"],"Notebook source changed during the run; start a new run.")
for relative,record in input_manifest.items():
    path=PROJECT_ROOT/relative; stat=path.stat()
    require(stat.st_size==record["bytes"] and stat.st_mtime_ns==record["modified_ns"],"Input changed during run: "+relative)
for name,table in tables.items(): table.to_csv(RUN_DIR/(name+".csv"),index=False)

def markdown_table(frame, columns):
    rows=["| "+" | ".join(columns)+" |", "| "+" | ".join(["---"]*len(columns))+" |"]
    for _,row in frame[columns].iterrows():
        values=[]
        for value in row:
            values.append(f"{value:.4f}" if isinstance(value,(float,np.floating)) else str(value))
        rows.append("| "+" | ".join(values)+" |")
    return "\n".join(rows)

comparisons=tables["paired_error_comparisons"]
overview=comparisons.loc[comparisons["scope"].eq("all")&comparisons["metric"].eq("brier")&comparisons["comparator"].isin(["p_baseline_unit","p_baseline_fire"])]
local=comparisons.loc[comparisons["scope"].eq("all")&comparisons["metric"].eq("brier")&comparisons["model"].eq("p_local_nine")&comparisons["comparator"].eq("p_shared")]
report=["# 08E — Reviewer diagnostics",f"Run: {RUN_DIR.name}. Smoke test: {IS_SMOKE_TEST}. Whole-fire resamples: {BOOTSTRAP_REPLICATES}.",
        "Exploratory follow-up designed after inspecting the historical results. No independent or prospective confirmation is claimed.",
        "## Fire concentration",markdown_table(tables["fire_size_concentration"],["scope","fires","fireline_units","largest_fire_share","top_10_unit_share"]),
        "## Matched historical baselines", "Negative model-minus-comparator differences favor the named model. The baseline and error averaging use matching weights.",
        markdown_table(overview,["scheme","model","weighting","model_error","comparator_error","difference","ci_lower","ci_upper"]),
        "## Local values versus shared fire/day values", "Both procedures use nine numerical families, identical tree capacity, equal-unit training weights, and identical fire splits. Shared summaries contain no future-anchor values or outcomes.",
        markdown_table(local,["scheme","weighting","model_error","comparator_error","difference","ci_lower","ci_upper"]),
        "## Selection diagnostics", "Inspect selection_diagnostic_status.csv before interpreting denominators. Candidate records are already restricted by upstream matching and processing. Full-record observation coverage is used only descriptively, never as a new predictor.",
        "## How to interpret the model comparisons",
        "- Simple local logistic tests a small fixed relationship using humidity, closest mapped fire-area distance, and recorded type. It is not the earlier full-feature elastic-net model.",
        "- A small local/shared difference would indicate little benefit from local variation under the tested representation; it would not prove that all skill is between fires.",
        "- The full-model comparison also changes predictor families. It is not a clean decomposition of SHAP attribution or physical effects.",
        "- Near-zero differences with intervals spanning zero do not establish equivalence. Do not select a new model or cutoff using these test results and then call the same results independent validation.",
        "## Limits", "Mapping timestamps are not verified construction times. Selection depends on recorded outcomes and later NIROPS observations. Shared summaries depend on the selected cohort. Repeated analysis, spatial dependence across fires, and uncertain historical predictor availability remain limitations.",
        "Intervals condition on fitted predictions and fixed analysis choices; they are pointwise, do not include retraining uncertainty, and are not adjusted for multiple comparisons.",
        "The geographic figure describes coverage and does not implement geographic blocking. No fuel-moisture product, PCL/SDI benchmark, construction-time validation, calibration fitting, or operational decision rule was added.",
        "## Files", "All tables are CSV; 08E_matched_predictions.parquet contains paired test predictions. models/ contains verified per-split checkpoints. figures/ contains PDF and PNG exports."
       ]
stages=tables["selection_stage_counts"]
if not stages.empty:
    report.extend(["## Saved selection stages",markdown_table(stages.loc[stages["scope"].eq("overall")],
        ["stage","candidate_fireline_units","retained_fireline_units","retained_fires","retained_unit_held_fraction"]),
        "## Final retention by recorded outcome",markdown_table(stages.loc[stages["scope"].eq("outcome")&stages["stage"].eq("final_modeling_cohort")],
        ["recorded_outcome","candidate_fireline_units","retained_fireline_units","retained_fraction_of_candidates"])])
convergence_issues=tables["fit_log"].loc[tables["fit_log"]["fit_convergence_review_required"].eq(True)]
report.extend(["## Fitting review",f"Fits requiring convergence review: {len(convergence_issues)} of 27. Consult fit_log.csv and model-specific fit_info.json before interpreting any flagged fit."])
if IS_SMOKE_TEST: report.insert(1,"**SOFTWARE VERIFICATION ONLY — do not report these model results in the paper.**")
(RUN_DIR/"08E_FINDINGS_REPORT.md").write_text("\n\n".join(report)+"\n",encoding="utf-8")
checks={"is_smoke_test":IS_SMOKE_TEST,"frozen_ids_labels_times_preserved":True,"fire_and_source_disjointness_checked":True,
        "forward_year_order_checked":True,"train_only_preprocessing_and_priors":True,"coarse_features_exclude_future_anchors":True,
        "unique_complete_paired_predictions":True,"all_27_checkpoints_present":len(fit_rows)==27,
        "input_file_stats_unchanged":True,"notebook_source_unchanged":True,"pointwise_intervals":True,
        "all_fits_passed_convergence_checks":len(convergence_issues)==0}
require(checks["all_27_checkpoints_present"],"Incomplete fitting plan.")
write_json(RUN_DIR/"08E_quality_checks.json",checks)
write_json(RUN_DIR/"08E_run_manifest.json",{"fingerprint":RUN_FINGERPRINT,"configuration":run_spec,"run_dir":str(RUN_DIR),
           "population_units":len(population),"population_fires":population["_irwin_norm"].nunique(),"tables":list(tables),"figures":figure_files,
           "python":sys.version,"platform":platform.platform(),"completed_utc":datetime.now(timezone.utc).isoformat()})
write_json(RUN_DIR/"08E_COMPLETE.json",{"fingerprint":RUN_FINGERPRINT,"is_smoke_test":IS_SMOKE_TEST,"complete":True})
display(Markdown("**08E completed.** Review package: `"+str(RUN_DIR)+"`"))
display(tables["paired_error_comparisons"].loc[lambda d:d["scheme"].eq("forward_year")&d["scope"].eq("all")&d["metric"].eq("brier")])

## Reading the results

1. Confirm that `08E_COMPLETE.json` exists and `is_smoke_test` is false for the scientific run.
2. Review the candidate and final observation-coverage denominators before discussing selection effects.
3. Compare the local-nine model with the shared model on the same rows. Read both pooled and within-fire ranking, along with both probability-error weightings.
4. Compare all models against the corresponding training-only historical baseline. Do not treat a test-year Held fraction as a predictive baseline.
5. Inspect annual reliability and fire concentration before interpreting a combined-year result.

These checks can strengthen or narrow the manuscript's claims. They do not determine the preferred scientific conclusion in advance.